## Загрузка датасета SOB-Hard на HuggingFace

Ноутбук собирает `test.json` и `shots.json` из `datasets/SOBHard2/` в `datasets.DatasetDict`
и заливает его на 🤗 Hub.

Что здесь специфично для этого датасета:

* поле `instruction` в локальных файлах — это **индекс** промпта в `dataset_meta.json["prompts"]`;
  перед заливкой он заменяется на сам текст промпта (общее правило MERA);
* `meta.reference` — эталонный документ, по которому считаются метрики содержания. Скоринг читает
  эталон именно оттуда, а не из `outputs`;
* `meta.checks` и `meta.task_meta` — JSON-строки с предвычисленными данными для верификаторов;
* `meta.dialogue_id` / `turn_index` / `n_turns` — координаты хода в многоходовом вопросе. Их читает
  сэмплер, который собирает диалог, и читает **до** разбора `task_meta`, поэтому они лежат
  отдельными полями. У одноходовых вопросов там `null`;
* перед заливкой прогоняется проверка целостности: каждый эталон обязан пройти все применимые
  ограничения тем же скорером, которым считается метрика.


In [1]:
import json
import os
import sys

import datasets
from tqdm import tqdm


c:\Users\arorlov\.conda\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Подготовка данных


#### WARNING!

Если ваш датасет является __ПРИВАТНЫМ__, оставьте `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS` равным `True`.
На ХФ даже приватно не должно лежать датасетов с ответами.

Задание здесь — выдать конкретный документ, и без эталона метрики содержания посчитать нечем.
Эталон лежит в `meta.reference`, поэтому стирание одного лишь `outputs` ответы **не прячет**.
Решение принимается явно, флагом `KEEP_REFERENCE_FOR_SCORING`. Ячейка «Что мы теряем» покажет
на числах цену каждого варианта.


In [12]:
MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS = True
KEEP_REFERENCE_FOR_SCORING = False


Пути указаны относительно расположения ноутбука в `datasets/SOBHard2/`.


In [3]:
path_to_data = "."
path_to_meta = "."

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


#### Подгрузка данных


In [5]:
shots = load_json(os.path.join(path_to_data, "shots.json"))["data"]
test = load_json(os.path.join(path_to_data, "test.json"))["data"]
meta = load_json(os.path.join(path_to_meta, "dataset_meta.json"))

print(f"shots: {len(shots)}, test: {len(test)}")


shots: 10, test: 825


In [6]:
prompts = meta["prompts"]
len(prompts)


20

#### Обработка полей датасета

На ХФ загружается датасет, где у КАЖДОГО сэмпла вместо числа в поле `instruction` стоит промпт.
Ячейка идемпотентна.


In [7]:
def resolve_prompts(split):
    for card in split:
        if isinstance(card["instruction"], int):
            card["instruction"] = prompts[card["instruction"]]


resolve_prompts(shots)
resolve_prompts(test)

print(test[0]["instruction"][:400])


Короче, есть задачка. Ниже всё написано — сделай и оформи как просят.

Задача:
{task}

Входные данные:
{input_data}

Формат ответа:
{format}

Инструкция:
{question}


#### Координаты многоходовых вопросов

У одноходовых вопросов полей диалога нет вовсе, а схема на Hub должна быть одинаковой для всех
строк. Проставляем `null` там, где их не было: сэмплер, собирающий диалог, трактует `null` как
«это не ход разговора» и не отдаёт такому вопросу никакой истории.


In [8]:
def normalise_dialogue_fields(split):
    for card in split:
        m = card["meta"]
        for key in ("dialogue_id", "turn_index", "n_turns"):
            m.setdefault(key, None)


normalise_dialogue_fields(shots)
normalise_dialogue_fields(test)

turns = [c for c in test if c["meta"]["dialogue_id"] is not None]
print("ходов диалога:", len(turns),
      "| диалогов:", len({c["meta"]["dialogue_id"] for c in turns}),
      "| одноходовых вопросов:", len(test) - len(turns))


ходов диалога: 75 | диалогов: 21 | одноходовых вопросов: 750


#### Проверка целостности перед заливкой

Подставляем `inputs` в `instruction` — промпт должен собираться без ошибок — и прогоняем эталон
через ту же библиотеку проверок, которой считается метрика. Эталон обязан набрать 1.0.

`balance_score` пропускается: на уровне одного вопроса это пара «клетка, зачёт», и смысл она
приобретает только после агрегации по всему прогону. Она проверяется отдельно, ниже.


In [9]:
sys.path.insert(0, os.path.abspath("../../benchmark_tasks/sobhard2"))
import utils as U

bad_prompt, bad_gold = [], []
for card in tqdm(shots + test):
    try:
        card["instruction"].format(**card["inputs"])
    except Exception as exc:
        bad_prompt.append((card["meta"]["id"], exc))
    m = U.process_results(card, [card["outputs"]])
    if any(v != 1.0 for v in U.scalar_metrics(m).values()):
        bad_gold.append((card["meta"]["id"], m))

print("промптов, которые не собираются:", len(bad_prompt))
print("эталонов, не набравших 1.0:", len(bad_gold))
assert not bad_prompt and not bad_gold

# И вторая метрика — на наборе целиком, а не по вопросу.
gold_balance = U.balance_score([U.process_results(c, [c["outputs"]])["balance_score"]
                                for c in test])
print("balance_score эталонного набора:", gold_balance)
assert abs(gold_balance - 1.0) < 1e-9


100%|██████████| 835/835 [00:06<00:00, 130.95it/s]


промптов, которые не собираются: 0
эталонов, не набравших 1.0: 0
balance_score эталонного набора: 1.0


#### Что мы теряем, если стереть эталон

Ячейка ничего не меняет: берёт копию данных, стирает `meta.reference` и показывает, сколько
проверок остаётся применимыми. Это и есть цена варианта `KEEP_REFERENCE_FOR_SCORING = False`.


In [10]:
import copy
from collections import Counter

probe = copy.deepcopy(test[:50])
for card in probe:
    card["meta"]["reference"] = ""
    card["outputs"] = ""

alive = Counter()
for card in probe:
    for ch in U.score_response(card, "```json\n{}\n```"):
        if ch["applicable"]:
            alive[ch["id"]] += 1

print("проверок остаётся применимыми (на выборке из 50 вопросов):")
for cid in U.constraint_ids():
    print(f"  {cid:32s} {alive.get(cid, 0):>3d}")


проверок остаётся применимыми (на выборке из 50 вопросов):
  pack.fence_present                50
  pack.fence_exactly_one            50
  pack.fence_tag_correct            50
  pack.nothing_outside_fence        50
  syn.parses_target                 50
  val.structure_match                0
  val.numeric_within_tol             0
  val.strings_byte_exact             0
  convert.lossless_vs_source         0
  extract.keys_exact                 0
  extract.is_object                  0
  extract.absent_paths_null          0
  repair.data_intact                 0
  transform.rule_applied             0
  transform.list_order_preserved     0
  transform.values_unchanged         0
  template.record_count              0
  template.no_placeholder_left       0
  template.rendered_exact            0
  applydiff.result_exact             0
  patch.ops_applied                  0
  merge.join_exact                   0
  merge.record_count                 0
  diff.matches_pair                  0
  dif

#### Убираем ответы для приватных задач

В `test` стираются ответы; в `shots` они сохраняются — это демонстрации, они и должны быть видны.


In [14]:
def hide_answers(dataset_split, drop_reference):
    for card in tqdm(dataset_split):
        card["outputs"] = ""
        if drop_reference:
            card["meta"]["reference"] = ""


if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS:
    hide_answers(test, drop_reference=not KEEP_REFERENCE_FOR_SCORING)

print("outputs стёрт:", all(c["outputs"] == "" for c in test)
      if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS else "outputs оставлен")
print("meta.reference на месте:", all(c["meta"]["reference"] for c in test))


100%|██████████| 825/825 [00:00<?, ?it/s]

outputs стёрт: True
meta.reference на месте: False


### Создаем датасет для загрузки на ХФ


#### Аннотация полей датасета

* `meta.task_meta` и `meta.checks` — именно **строки** с JSON внутри: у разных семейств разный
  набор параметров, фиксированной схемой их не описать;
* `meta.reference` — сырой текст эталонного документа, а не разобранный JSON. Хранить разобранным
  нельзя: в YAML и TOML ключи отображений бывают не строками, и JSON-сериализация превратила бы их
  в строки только с одной стороны сравнения;
* поля диалога объявлены как целые и допускают `null` — у одноходовых вопросов их нет.


In [17]:
features = datasets.Features({
    "instruction": datasets.Value("string"),
    "inputs": {
        "task": datasets.Value("string"),
        "input_data": datasets.Value("string"),
        "format": datasets.Value("string"),
        "question": datasets.Value("string"),
    },
    "outputs": datasets.Value("string"),
    "meta": {
        "id": datasets.Value("int32"),
        "base_id": datasets.Value("string"),
        "fence_tag": datasets.Value("string"),
        "reference": datasets.Value("string"),
        "reference_sha256": datasets.Value("string"),
        "task_meta": datasets.Value("string"),
        "checks": datasets.Value("string"),
        # Координаты хода в разговоре; null у одноходовых вопросов.
        "dialogue_id": datasets.Value("int32"),
        "turn_index": datasets.Value("int32"),
        "n_turns": datasets.Value("int32"),
        "categories": {
            "family": datasets.Value("string"),
            "difficulty": datasets.Value("string"),
            "language": datasets.Value("string"),
            "source_format": datasets.Value("string"),
            "target_format": datasets.Value("string"),
            "length_tier": datasets.Value("string"),
            "prompt_style": datasets.Value("string"),
            # Происхождение: из какого файла корпуса взят документ.
            "origin": datasets.Value("string"),
        },
    },
})


#### Создание датасетов для каждого сплита


In [18]:
shots_ds = datasets.Dataset.from_list(shots, features=features)
test_ds = datasets.Dataset.from_list(test, features=features)
shots_ds, test_ds


(Dataset({
     features: ['instruction', 'inputs', 'outputs', 'meta'],
     num_rows: 10
 }),
 Dataset({
     features: ['instruction', 'inputs', 'outputs', 'meta'],
     num_rows: 825
 }))

##### Проверка

Ничего не потеряно, не продублировано и не переехало — включая порядок ходов в диалогах.


In [19]:
from collections import Counter

# количество вопросов до конвертации и после совпадает
assert len(test) == len(test_ds) and len(shots) == len(shots_ds)

# id вопросов сходятся и остаются сквозными
assert [c["meta"]["id"] for c in test] == [c["meta"]["id"] for c in test_ds]
assert [c["meta"]["id"] for c in shots] == [c["meta"]["id"] for c in shots_ds]
ids = sorted(c["meta"]["id"] for c in shots + test)
assert ids == list(range(1, len(ids) + 1))

# сетка: в каждой клетке ровно 25 вопросов
cells = Counter((c["meta"]["categories"]["family"],
                 c["meta"]["categories"]["difficulty"]) for c in test_ds)
assert set(cells.values()) == {25}, cells
print("клеток:", len(cells), "| вопросов в клетке: 25")

# meta.checks и meta.task_meta всё ещё парсятся
assert all(json.loads(c["meta"]["checks"]) is not None for c in test_ds)
assert all(json.loads(c["meta"]["task_meta"]) is not None for c in test_ds)

# диалоги пережили конвертацию: ходы идут подряд, по порядку, в одном сплите
by_dialogue = {}
for c in test_ds:
    did = c["meta"]["dialogue_id"]
    if did is not None:
        by_dialogue.setdefault(did, []).append(c["meta"])
for did, turns in by_dialogue.items():
    idx = sorted(t["turn_index"] for t in turns)
    assert idx == list(range(1, len(idx) + 1)), (did, idx)
    assert {t["n_turns"] for t in turns} == {len(idx)}, (did, turns)
assert not any(c["meta"]["dialogue_id"] is not None for c in shots_ds), \
    "ход диалога попал в shots — история собирается только внутри test"
print("диалогов:", len(by_dialogue), "| ходов:", sum(len(v) for v in by_dialogue.values()))
print("OK")


клеток: 33 | вопросов в клетке: 25
диалогов: 21 | ходов: 75
OK


#### Собираем сплиты в один датасет


In [20]:
dataset = datasets.DatasetDict({"shots": shots_ds, "test": test_ds})
dataset


DatasetDict({
    shots: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 10
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 825
    })
})

### Загрузка датасета на ХФ

Понадобятся токен с правом записи и путь для записи. Название пишите ровно так, как оно заявлено
в мете (`dataset_meta.json["dataset_name"]`), регистр имеет значение.

Советуем сначала залить приватно и выслать на mera@a-ai.ru токен и путь для верификации.


In [ ]:
from dotenv import load_dotenv

load_dotenv('../../.env')

### TOKEN
token = os.getenv('HF_TOKEN')
if token is None:
    raise ValueError("HF_TOKEN not found in .env file")

### UPLOAD PATH
HF_REPO_ID = "MERA-evaluation/SOBHard"

# Предварительный просмотр — залейте в свой приватный репозиторий:
# HF_REPO_ID = "<your-account>/SOBHard"

### PRIVATE OR PUBLIC
upload_private = True

print("upload to:", HF_REPO_ID, "| private:", upload_private)


upload to: MERA-evaluation/SOBHard | private: True


In [24]:
dataset.push_to_hub(HF_REPO_ID, private=upload_private, token=token)

Setting num_proc from 1 back to 1 for the shots split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 228.58ba/s]
Processing Files (1 / 1): 100%|██████████|  132kB /  132kB, 60.2kB/s  
New Data Upload: 100%|██████████|  132kB /  132kB, 60.2kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.42s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 36.04ba/s]
Processing Files (1 / 1): 100%|██████████| 36.0MB / 36.0MB, 12.8MB/s  
New Data Upload: 100%|██████████| 33.5MB / 33.5MB, 11.9MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.84s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/MERA-evaluation/SOBHard/commit/e88356346ac276383e23fdaab3430bb137c942ce', commit_message='Upload dataset', commit_description='', oid='e88356346ac276383e23fdaab3430bb137c942ce', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MERA-evaluation/SOBHard', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MERA-evaluation/SOBHard'), pr_revision=None, pr_num=None)

### Проверка того, как датасет загрузился на ХФ

Загрузим датасет обратно и убедимся, что его увидит корректно любой, кто его скачает.


In [25]:
reloaded = datasets.load_dataset(HF_REPO_ID, token=token)
reloaded

c:\Users\arorlov\.conda\envs\default\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arorlov\.cache\huggingface\hub\datasets--MERA-evaluation--SOBHard. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 825/825 [00:00<00:00, 18670.22 examples/s]


DatasetDict({
    shots: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 10
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 825
    })
})

In [26]:
# сплиты, размеры и сетка совпадают с локальными
assert len(reloaded["test"]) == len(test) and len(reloaded["shots"]) == len(shots)
back = Counter((c["meta"]["categories"]["family"],
                c["meta"]["categories"]["difficulty"]) for c in reloaded["test"])
assert back == cells

# диалоги доехали целыми
back_turns = [c for c in reloaded["test"] if c["meta"]["dialogue_id"] is not None]
assert len(back_turns) == sum(len(v) for v in by_dialogue.values())

# и скорер по-прежнему ставит эталону 1.0 — там, где эталон остался
if all(c["meta"]["reference"] for c in reloaded["test"]):
    sample = list(reloaded["test"])[:25]
    for card in sample:
        gold = f"```{card['meta']['fence_tag']}\n{card['meta']['reference']}\n```"
        m = U.scalar_metrics(U.process_results(card, [gold]))
        assert all(v == 1.0 for v in m.values()), (card["meta"]["id"], m)
    print("эталон на Hub по-прежнему набирает 1.0")
else:
    print("эталон стёрт — content_pass_rate и balance_score на Hub не считаются")
print("OK")


эталон стёрт — content_pass_rate и balance_score на Hub не считаются
OK
